# 01 - Deep Agents 生产部署

**来源:** [LangChain Docs - Going to Production](https://docs.langchain.com/oss/python/deepagents/going-to-production)

本指南涵盖将 Deep Agent 从本地原型推向生产环境的关键考量：持久化记忆、执行环境、安全护栏和部署选项。

## 1. LangSmith 部署

推荐方案：**Managed Deep Agents** - 在 LangSmith 中创建、运行和运维 Deep Agent 的 API-first 托管运行时。

为需要自定义应用代码、自定义路由或高级认证的团队，也可以通过 **LangSmith Deployment** 直接配置。

部署配置文件 `langgraph.json`：

| 字段 | 说明 |
|------|------|
| dependencies | 依赖包，["."] 安装当前目录(读取 requirements.txt / pyproject.toml) |
| graphs | 映射 Graph ID 到代码位置 "<id>": "./<file>:<variable>" |
| env | .env 文件路径(API keys、secrets) |

In [ ]:
# langgraph.json 示例
{
  "dependencies": ["."],
  "graphs": {
    "agent": "./agent.py:agent"
  },
  "env": ".env"
}

## 2. 调用 Agent(Invocation)

生产环境中每次调用应携带两个运行时参数：

| 参数 | 位置 | 说明 |
|------|------|------|
| thread_id | config.configurable.thread_id | 对话标识符，Checkpointer 用它持久化消息历史 |
| context | invoke(context=...) | 每次运行的数据(user_id, API keys, feature flags 等) |

In [ ]:
from dataclasses import dataclass
from deepagents import create_deep_agent
from langchain_core.utils.uuid import uuid7

@dataclass
class Context:
    user_id: str

agent = create_deep_agent(
    model="deepseek-v4-flash",
    context_schema=Context,
)

# 开始新对话
config = {"configurable": {"thread_id": str(uuid7())}}
agent.invoke(
    {"messages": [{"role": "user", "content": "Plan a 3-day trip to Tokyo"}]},
    config=config,
    context=Context(user_id="user-123"),
)

# 继续同一对话：复用同一 thread_id
agent.invoke(
    {"messages": [{"role": "user", "content": "Make it 5 days instead"}]},
    config=config,
    context=Context(user_id="user-123"),
)

## 3. 多租户与认证

在需要为用户隔离数据的环境中，有三种作用域：

| 作用域 | 说明 |
|--------|------|
| Thread | 单个对话。消息历史和文件默认局限在线程内 |
| User | 用户级别。记忆和文件可私有或跨用户共享 |
| Assistant | Agent 实例级别。记忆和文件可绑定到单个实例或全局共享 |

### 在 Tool 中读取认证信息

通过 Runtime Context 读取当前用户信息：

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
async def github_action(runtime: ToolRuntime):
    """Run a GitHub Action on behalf of the current user"""
    user_id = runtime.context.user_id
    auth_token = runtime.context.get("auth_token")
    print(f"Running action for user {user_id}")

## 4. 执行环境配置

| 场景 | 推荐 Backend |
|------|--------------|
| 简单对话，无需文件持久化 | StateBackend |
| 需要文件持久化 | FilesystemBackend |
| 需要 LangGraph Store 持久化 | StoreBackend(store后台) |
| 需要远程沙箱执行代码 | ModalSandbox / DaytonaSandbox / RunloopSandbox |
| 需要多重存储 | CompositeBackend(分层路由) |

### CompositeBackend 示例

In [ ]:
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend

backend = CompositeBackend()
backend.add_backend(StateBackend(), prefix="/state/")     # 工作区: 内存
backend.add_backend(StoreBackend(store), prefix="/store/")  # 持久化存储

agent = create_deep_agent(
    model="deepseek-v4-flash",
    backend=backend,
)

## 5. 安全护栏

| 措施 | 实现方式 |
|------|----------|
| 速率限制 | 通过 LangSmith Engine 或 API Gateway 配置 |
| 错误处理 | RetryMiddleware + fallback_providers |
| 数据隐私 | 输入校验、输出过滤、PII 脱敏 |
| 权限控制 | FilesystemPermission 声明式文件访问规则 |

## 6. LangGraph SDK 调用

当通过 LangSmith 部署时，SDK 会为你管理线程：

In [ ]:
from langgraph_sdk import get_client

client = get_client(url="<DEPLOYMENT_URL>", api_key="<LANGSMITH_API_KEY>")
thread = await client.threads.create()

async for chunk in client.runs.stream(
    thread["thread_id"],
    "agent",
    input={"messages": [{"role": "user", "content": "Hello"}]},
):
    print(chunk.data)

## 7. 前端集成

部署后的 Agent 可通过 LangGraph SDK 的 `client.runs.stream()` 进行流式调用。推荐结合 `useStream` React Hook 在浏览器中消费。

详见 [Frontend Overview](https://docs.langchain.com/oss/python/deepagents/frontend/overview)。

---
**参考链接**

- [Going to Production - LangChain Docs](https://docs.langchain.com/oss/python/deepagents/going-to-production)
- [Backends](https://docs.langchain.com/oss/python/deepagents/backends)
- [Managed Deep Agents](https://docs.langchain.com/langsmith/deploy-managed-deep-agent)
- [Frontend Overview](https://docs.langchain.com/oss/python/deepagents/frontend/overview)